# Deep Learning Fundamentals: Computational Graphs, Backprop & Optimizers

This interactive lab covers the essential mechanics of Deep Learning:
1. **Forward & Backward Propagation**: Computational graphs and manual analytical backpropagation.
2. **From-Scratch Layer Implementation**: `DenseLayer`, `ReLU`, and `SoftmaxCrossEntropyLoss`.
3. **First-Order Optimization**: Comparing SGD, Momentum, RMSprop, and AdamW.
4. **PyTorch Reference Implementation**: Training a multi-layer perceptron on non-linear synthetic data.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd() / 'code'))

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from dl_fundamentals import (
    DenseLayer, ReLU, SoftmaxCrossEntropyLoss,
    SGDMomentum, RMSprop, AdamW, ReferencePyTorchMLP
)

print(f'PyTorch Version: {torch.__version__}')
print('Deep Learning Fundamentals environment initialized successfully!')

## 1. Scratch 2-Layer Neural Network with Analytical Backprop

We synthesize non-linear classification data (XOR-like clusters) and train a 2-layer MLP using our modular NumPy layers.

In [ ]:
np.random.seed(42)
N = 300
# Generate two concentric rings
r1 = np.random.uniform(0.0, 1.0, size=(N // 2, 1))
theta1 = np.random.uniform(0, 2 * np.pi, size=(N // 2, 1))
X1 = np.hstack([r1 * np.cos(theta1), r1 * np.sin(theta1)])

r2 = np.random.uniform(1.5, 2.5, size=(N // 2, 1))
theta2 = np.random.uniform(0, 2 * np.pi, size=(N // 2, 1))
X2 = np.hstack([r2 * np.cos(theta2), r2 * np.sin(theta2)])

X = np.vstack([X1, X2])
y = np.array([0] * (N // 2) + [1] * (N // 2))

# Build network: 2 -> 16 -> 2
layer1 = DenseLayer(2, 16, init_method='he')
act1 = ReLU()
layer2 = DenseLayer(16, 2, init_method='xavier')
loss_fn = SoftmaxCrossEntropyLoss()
optimizer = AdamW([layer1, layer2], lr=0.03, weight_decay=1e-4)

losses = []
for epoch in range(100):
    optimizer.zero_grad()
    # Forward
    h1 = act1.forward(layer1.forward(X))
    logits = layer2.forward(h1)
    loss = loss_fn.forward(logits, y)
    losses.append(loss)
    # Backward
    d_logits = loss_fn.backward()
    d_h1 = layer2.backward(d_logits)
    d_z1 = act1.backward(d_h1)
    layer1.backward(d_z1)
    # Update
    optimizer.step()

preds = np.argmax(logits, axis=1)
acc = np.mean(preds == y)
print(f'Scratch MLP Initial Loss: {losses[0]:.4f} -> Final Loss: {losses[-1]:.4f}')
print(f'Scratch MLP Classification Accuracy: {acc * 100:.1f}%')

## 2. Optimizer Convergence Comparison

We compare training loss trajectories across different first-order optimization algorithms on the same architecture.

In [ ]:
opts_to_test = {
    'SGD + Momentum': lambda l: SGDMomentum(l, lr=0.05, momentum=0.9),
    'RMSprop': lambda l: RMSprop(l, lr=0.01),
    'AdamW': lambda l: AdamW(l, lr=0.03, weight_decay=1e-4)
}

for name, opt_fn in opts_to_test.items():
    np.random.seed(42)
    l1 = DenseLayer(2, 16, init_method='he')
    a1 = ReLU()
    l2 = DenseLayer(16, 2, init_method='xavier')
    opt = opt_fn([l1, l2])
    
    curr_losses = []
    for _ in range(80):
        opt.zero_grad()
        out = l2.forward(a1.forward(l1.forward(X)))
        loss = loss_fn.forward(out, y)
        curr_losses.append(loss)
        d_out = loss_fn.backward()
        l1.backward(a1.backward(l2.backward(d_out)))
        opt.step()
    print(f'{name:<16} | Initial: {curr_losses[0]:.4f} | Final: {curr_losses[-1]:.4f}')

## 3. Production PyTorch Implementation

We implement the same problem using PyTorch `nn.Module` and evaluate gradient flow and parameters.

In [ ]:
X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.long)

torch.manual_seed(42)
torch_model = ReferencePyTorchMLP(in_features=2, hidden_dim=16, out_features=2, dropout_rate=0.0)
criterion = nn.CrossEntropyLoss()
pt_optimizer = optim.AdamW(torch_model.parameters(), lr=0.03, weight_decay=1e-4)

for epoch in range(100):
    pt_optimizer.zero_grad()
    logits = torch_model(X_t)
    loss = criterion(logits, y_t)
    loss.backward()
    pt_optimizer.step()

with torch.no_grad():
    pt_preds = torch.argmax(torch_model(X_t), dim=1)
    pt_acc = (pt_preds == y_t).float().mean().item()

print(f'PyTorch MLP Final Loss: {loss.item():.4f}')
print(f'PyTorch MLP Accuracy:   {pt_acc * 100:.1f}%')
assert pt_acc > 0.90, 'PyTorch model did not achieve expected non-linear accuracy!'